# 05 – Métricas do Lote 4 (L4)

HIS-01, HIS-02, HIS-05, HIS-06, HIS-07, HIS-08, HIS-09, HIS-10, ADD-01 e ADD-02, segundo `spec/metodologia_metricas.md` (secções 7B e 7D).

- **Não lê `processed/`**: lê as tabelas do Lote 1 que já estão em `metrics/` (`l1_cal_monthly`, `l1_cal_annual`, `l1_obs01_raw_counts`, `l1_cal_coverage`). Corra antes o notebook `02_metrics_lote1`.
- As tabelas são pequenas: não há ciclo por ano nem retoma; cada execução recalcula tudo. Com um mundo do tamanho esperado (170 países × 3 séries × 25 anos, cerca de 150 mil linhas mensais) demora aproximadamente 1 minuto, quase todo em HIS-05, HIS-06 e HIS-09.
- Escreve `metrics/l4_*.parquet`, `metrics/_log_lote4.csv` e `metrics/_params_lote4.json`.
- O código vem do repo (`git clone` do GitHub público): `data/pipeline/metrics/`.

**Antes de correr:** o atalho `Sprinklers` tem de estar em *O Meu Drive* (ver `data/README.md`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Sprinklers')        # ajustar se o atalho tiver outro nome
METRICS = ROOT / 'metrics'
REPO_URL = 'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git'
REPO = Path('/content/repo')


def ensure_repo():
    if (REPO / '.git').exists():
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)


ensure_repo()
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))

REQUIRED = ['l1_cal_monthly', 'l1_cal_annual', 'l1_obs01_raw_counts', 'l1_cal_coverage']
missing = [n for n in REQUIRED if not (METRICS / f'{n}.parquet').exists()]
if missing:
    raise FileNotFoundError(f'Faltam em {METRICS}: {missing}. Corra primeiro o notebook 02_metrics_lote1.')

In [ ]:
# Parametros do lote (valores por defeito da secao 7B do documento de metodologia)
REF_START = None          # primeiro ano da referencia (None = sem limite)
REF_END = None            # ultimo ano da referencia (None = sem limite)
MIN_REF_YEARS = 5         # abaixo disto as estatisticas ficam NA
MAX_UNKNOWN_DAYS = 0      # dias de cobertura desconhecida tolerados num mes/ano valido
MIN_TREND_YEARS = 8       # minimo de anos para a tendencia (HIS-10)

In [ ]:
import json, datetime
import pandas as pd

from metrics.history_distribution import (
    his01_hist_median, his02_hist_band, his05_zscore, his06_percentile, his09_months_above,
)
from metrics.history_cumulative import his07_ytd, his08_yoy, his10_trend
from metrics.coverage_meta import add01_active_satellites, add02_zero_days


def atomic_write(df, path):
    tmp = path.with_name(path.name + '.tmp')
    df.to_parquet(tmp, index=False)
    tmp.replace(path)


monthly = pd.read_parquet(METRICS / 'l1_cal_monthly.parquet')
annual = pd.read_parquet(METRICS / 'l1_cal_annual.parquet')
obs01 = pd.read_parquet(METRICS / 'l1_obs01_raw_counts.parquet')
coverage = pd.read_parquet(METRICS / 'l1_cal_coverage.parquet')
print({n: len(t) for n, t in {'monthly': monthly, 'annual': annual, 'obs01': obs01, 'coverage': coverage}.items()})

ref = dict(ref_start=REF_START, ref_end=REF_END, min_ref_years=MIN_REF_YEARS, max_unknown_days=MAX_UNKNOWN_DAYS)
tables = {
    'l4_his01_hist_median': his01_hist_median(monthly, **ref),
    'l4_his02_hist_band': his02_hist_band(monthly, **ref),
    'l4_his05_zscore': his05_zscore(monthly, **ref),
    'l4_his06_percentile': his06_percentile(monthly, **ref),
    'l4_his07_ytd': his07_ytd(monthly, **ref),
    'l4_his08_yoy': his08_yoy(monthly, max_unknown_days=MAX_UNKNOWN_DAYS),
    'l4_his09_months_above': his09_months_above(monthly, **ref),
    'l4_his10_trend': his10_trend(annual, max_unknown_days=MAX_UNKNOWN_DAYS, min_years=MIN_TREND_YEARS),
    'l4_add01_active_satellites': add01_active_satellites(obs01),
    'l4_add02_zero_days': add02_zero_days(coverage),
}
for name, tbl in tables.items():
    atomic_write(tbl, METRICS / f'{name}.parquet')
    print(f'{name}: {len(tbl)} linhas')

try:
    commit = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
except Exception:
    commit = ''
params = dict(ref_start=REF_START, ref_end=REF_END, min_ref_years=MIN_REF_YEARS,
              max_unknown_days=MAX_UNKNOWN_DAYS, min_trend_years=MIN_TREND_YEARS, repo_commit=commit,
              run_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'))
(METRICS / '_params_lote4.json').write_text(json.dumps(params, indent=1))
log = pd.DataFrame([dict(table=n, n_rows=len(t), **params) for n, t in tables.items()])
log.to_csv(METRICS / '_log_lote4.csv', index=False)
display(log[['table', 'n_rows']])

## Verificação
Confirma invariantes entre as tabelas (falha com `AssertionError` se alguma não bater).

In [ ]:
z, p = tables['l4_his05_zscore'], tables['l4_his06_percentile']
assert len(z) == len(monthly) == len(p), 'HIS-05/06 devem ter uma linha por linha de l1_cal_monthly'
pc = p['percentile'].dropna()
assert pc.between(0, 100).all(), 'percentil fora de [0, 100]'
h9 = tables['l4_his09_months_above']
assert (h9['n_months_above'] <= h9['n_months_evaluated']).all()
b = tables['l4_his02_hist_band'].dropna(subset=['p10', 'p90'])
assert (b['p10'] <= b['p90']).all(), 'p10 > p90'
a2 = tables['l4_add02_zero_days']
assert a2['pct_zero_days'].between(0, 100).all() and (a2['n_zero_days'] <= a2['n_days']).all()
y = tables['l4_his08_yoy']
assert (y.loc[~y['valid_pair'], 'yoy_pct'].isna()).all(), 'yoy_pct so existe para pares validos'
print('OK: invariantes verificadas')